# A1 Plate Detection — Training Notebook (YOLOv8 / YOLO26)

**UIT Graduation Thesis 2026** — Smart Parking Management System using Computer Vision and Edge AI

Trains 2 model families (`yolov8n`, `yolo26n`) across 3 seeds on the A1 dataset (2-class: `bien_1hang`, `bien_2hang`).  
All logic lives in the `plate_detect` package; this notebook only orchestrates.

**How to run:** Select a Colab GPU runtime → Run All. Resumable via Drive checkpoints (each seed writes to Drive; skip already-completed seeds by re-running from cell 9).

In [ ]:
# ── Config — edit here, then Run All ────────────────────────────────────────
DRIVE_ROOT = "/content/drive/MyDrive/UIT_2025"
REPO_URL   = "https://github.com/UIT-DoAnCuoiKi/UIT2026-DoAnCuoiKi.git"
MODELS     = ["yolov8n", "yolo26n"]
SEEDS      = [0, 1, 2]

In [ ]:
# ── 1. Pinned install ─────────────────────────────────────────────────────
!pip install -q ultralytics==8.4.37
import ultralytics
ultralytics.checks()

In [ ]:
# ── 2. GPU check ──────────────────────────────────────────────────────────
import torch
assert torch.cuda.is_available(), "Select a Colab GPU runtime (not TPU/CPU)."
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ── 3. Clone repo + install plate_detect package ─────────────────────────
import os
if not os.path.exists("/content/repo"):
    !git clone -q $REPO_URL /content/repo
!pip install -q -e /content/repo/src/ml/plate_detect

In [ ]:
# ── 4. Mount Drive (checkpoints + weights land here) ─────────────────────
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(f"{DRIVE_ROOT}/runs", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/weights", exist_ok=True)
print("Drive mounted. Runs →", f"{DRIVE_ROOT}/runs")

In [ ]:
# ── 5. Pull A1 dataset via Kaggle API ────────────────────────────────────
# Requires ~/.kaggle/kaggle.json — upload it via the Colab sidebar first, e.g.:
#   from google.colab import files; files.upload()  # then move to ~/.kaggle/
import os
os.makedirs("/root/.kaggle", exist_ok=True)
# Uncomment and run once if kaggle.json was just uploaded:
# import shutil; shutil.copy("/content/kaggle.json", "/root/.kaggle/kaggle.json")
# !chmod 600 /root/.kaggle/kaggle.json

!kaggle datasets download -d duydieunguyen/licenseplates \
    -p /content/data/raw/kaggle_vn_plate_segment --unzip
print("Dataset ready at /content/data/raw/kaggle_vn_plate_segment")

In [ ]:
# ── 6. Build Config + prepare dataset ────────────────────────────────────
from plate_detect.config import Config
from plate_detect.data.prepare import prepare

cfg = Config(
    raw_dir="/content/data/raw/kaggle_vn_plate_segment",
    processed_dir="/content/data/processed/a1_det",
    dataset_yaml="/content/repo/src/ml/plate_detect/configs/a1_det.yaml",
    split_dir="/content/repo/src/ml/plate_detect/configs/split",
)
result = prepare(cfg)
print("prepare →", result)

In [ ]:
# ── 7. Train MODELS × SEEDS, evaluate on test split, record metrics ────────
import os, shutil
from ultralytics import YOLO
from plate_detect.train.trainer import run_train
from plate_detect.eval.evaluate import aggregate_seeds, append_experiment

FIGURES_DIR = "/content/repo/docs/report/figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

runs = {}
for mk in MODELS:
    seed_metrics = []
    for s in SEEDS:
        print(f"\n{'='*60}")
        print(f"Training {mk}  seed={s}")
        print(f"{'='*60}")
        # run_train returns the save_dir string (Drive-backed via project=)
        rd = run_train(mk, cfg, cfg.dataset_yaml, seed=s, project=f"{DRIVE_ROOT}/runs")
        best_pt = os.path.join(rd, "weights", "best.pt")

        # evaluate on the held-out test split
        m = YOLO(best_pt).val(data=cfg.dataset_yaml, split="test")
        metrics = {
            "map50":     m.box.map50,
            "map5095":   m.box.map,
            "precision": m.box.mp,
            "recall":    m.box.mr,
        }
        seed_metrics.append(metrics)

        # save eval figures for thesis (confusion_matrix.png, results.png)
        for fig_name in ("confusion_matrix.png", "results.png", "PR_curve.png"):
            src = os.path.join(str(m.save_dir), fig_name)
            if os.path.exists(src):
                dst = os.path.join(FIGURES_DIR, f"{mk}_s{s}_{fig_name}")
                shutil.copy(src, dst)
                print(f"  saved figure → {dst}")

        print(f"  mAP@0.5={metrics['map50']:.4f}  mAP@0.5:0.95={metrics['map5095']:.4f}")

    agg = aggregate_seeds(seed_metrics)
    runs[mk] = agg
    best_seed_idx = max(range(len(SEEDS)), key=lambda i: seed_metrics[i]["map50"])
    best_metrics  = seed_metrics[best_seed_idx]
    append_experiment(
        "/content/repo/src/ml/experiments.csv",
        model=mk,
        dataset="A1",
        hyperparams=f"imgsz={cfg.imgsz};epochs={cfg.epochs};seeds={SEEDS}",
        m=best_metrics,
        weights=f"weights/{mk}_a1_s{best_seed_idx}.pt",
    )
    print(f"\n{mk} aggregated: {agg}")

print("\nAll runs complete.")
print(runs)

In [ ]:
# ── 8. Export best model per family to ONNX ─────────────────────────────
import os, shutil, glob
from plate_detect.export.to_onnx import export

WEIGHTS_DIR = f"{DRIVE_ROOT}/weights"
os.makedirs(WEIGHTS_DIR, exist_ok=True)

for mk in MODELS:
    # find best checkpoint across seeds: highest mAP@0.5 run dir
    run_dirs = sorted(glob.glob(f"{DRIVE_ROOT}/runs/{mk}_s*"))
    if not run_dirs:
        print(f"WARNING: no run dirs found for {mk} — skipping export")
        continue

    best_pt = None
    best_map50 = -1.0
    for rd in run_dirs:
        pt = os.path.join(rd, "weights", "best.pt")
        if not os.path.exists(pt):
            continue
        m = __import__("ultralytics", fromlist=["YOLO"]).YOLO(pt).val(
            data=cfg.dataset_yaml, split="test", verbose=False
        )
        if m.box.map50 > best_map50:
            best_map50 = m.box.map50
            best_pt = pt

    if best_pt is None:
        print(f"WARNING: no valid best.pt found for {mk}")
        continue

    out_pt   = os.path.join(WEIGHTS_DIR, f"{mk}_a1_best.pt")
    out_onnx = os.path.join(WEIGHTS_DIR, f"{mk}_a1_best.onnx")
    shutil.copy(best_pt, out_pt)

    exported = export(weights_pt=best_pt, out_onnx=out_onnx, imgsz=cfg.imgsz)
    print(f"{mk}: best seed mAP@0.5={best_map50:.4f}")
    print(f"  .pt  → {out_pt}")
    print(f"  .onnx→ {exported}")

print("\nExport complete. Copy weights to src/ml/plate_detect/weights/ (git-lfs) from your local machine.")

## Next Steps — copy weights and commit (from your local machine)

After the training run finishes:

1. **Download weights from Drive** (`$DRIVE_ROOT/weights/`):
   - `yolov8n_a1_best.pt` + `yolov8n_a1_best.onnx`
   - `yolo26n_a1_best.pt` + `yolo26n_a1_best.onnx`

2. **Place them in `src/ml/plate_detect/weights/`** (tracked by git-lfs — see `.gitattributes`).

3. **Commit from your local machine:**
   ```bash
   git add src/ml/plate_detect/weights/
   git add src/ml/experiments.csv
   git add docs/report/figures/
   git commit -m "feat(plate_detect): add trained A1 weights and eval figures"
   git push
   ```

**Success gate:** mAP@0.5 >= 0.90 on the test split for both model families.  
If below threshold: increase `epochs` in `Config`, add more augmentation, or collect more data.